# 02 — Feature Engineering

Builds the full model-ready dataset from raw data.

**Steps:**
1. Extract raw .7z files
2. Load station status (12 months, aggregated to hourly)
3. Load station info and compute distance features
4. Fetch weather from Open-Meteo
5. Add temporal and day-type features
6. Compute occupancy and lag features
7. Compute hist_mean_occ on training rows only (no leakage)
8. Expand to all horizons — one row per (station, T, horizon)
9. Export to `data/processed/bicing_model_dataset.parquet`

**Output:** `data/processed/bicing_model_dataset.parquet`

In [ ]:
import sys
from pathlib import Path

# Make src/ importable from the notebooks/ directory
PROJECT_ROOT = Path('..').resolve()
sys.path.append(str(PROJECT_ROOT))

import gc
import pandas as pd
import numpy as np

from src.data.loader import extract_all_7z, load_station_info, load_station_status, fetch_weather
from src.data.features import (
    add_occupancy, add_temporal_features, add_day_type,
    add_distance_features, add_lag_features,
    add_historical_baseline, add_cyclic_features, expand_horizons
)
from src.utils.config import PATHS, SPLIT_DATE, HORIZONS

# Define output path relative to the project root so there's no ambiguity
# regardless of where Jupyter resolves __file__
OUTPUT_PATH = PROJECT_ROOT / 'data' / 'processed' / 'bicing_model_dataset.parquet'
print(f'Output will be saved to:\n  {OUTPUT_PATH}')

---
## Step 1 — Extract raw files

In [ ]:
extract_all_7z()

---
## Step 2 — Load station status (hourly)

In [ ]:
df_hourly = load_station_status()
print(df_hourly.shape)
df_hourly.head(3)

---
## Step 3 — Station info and distances

In [ ]:
df_stations = load_station_info()
df_stations = add_distance_features(df_stations)
print(df_stations[['name', 'capacity', 'dist_beach', 'dist_center']].head(5))

In [ ]:
df_hourly['station_id'] = df_hourly['station_id'].astype(int)
df_full = df_hourly.merge(df_stations, on='station_id', how='left')

unmatched = df_full['lat'].isnull().sum()
print(f'Rows without station info: {unmatched} ({unmatched/len(df_full)*100:.2f}%)')
df_full = df_full.dropna(subset=['lat'])

del df_hourly
gc.collect()

print(f'Shape after merge: {df_full.shape}')

---
## Step 4 — Occupancy

In [ ]:
df_full = add_occupancy(df_full)
print(f'After occupancy: {df_full.shape}')
print(f'Occupancy range: {df_full["occupancy"].min():.3f} — {df_full["occupancy"].max():.3f}')

---
## Step 5 — Weather

In [ ]:
start = str(df_full['datetime_hour'].min().date())
end   = str(df_full['datetime_hour'].max().date())
df_weather = fetch_weather(start, end)

df_full = df_full.merge(df_weather, on='datetime_hour', how='left')
del df_weather
gc.collect()

print(f'Weather nulls: {df_full["temperature"].isnull().sum()}')

---
## Step 6 — Temporal features and day type

In [ ]:
df_full = add_temporal_features(df_full)
df_full = add_day_type(df_full)
df_full = add_cyclic_features(df_full)

print(df_full[['hour', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'is_weekend', 'is_holiday']].head(3))

---
## Step 7 — Lag features

In [ ]:
df_full = df_full.sort_values(['station_id', 'datetime_hour']).reset_index(drop=True)
df_full = add_lag_features(df_full)

before = len(df_full)
df_full = df_full.dropna(subset=['lag_168h', 'rolling_mean_3h'])
print(f'Dropped {before - len(df_full):,} rows (insufficient history). Remaining: {len(df_full):,}')

---
## Step 8 — Temporal split and hist_mean_occ (leakage-safe)

The historical baseline is computed **only on training rows** and then
joined to the full dataset. Computing it on the full dataset would let
the model see test-set occupancy patterns through this feature.

In [ ]:
train_mask = df_full['datetime_hour'] < SPLIT_DATE
print(f'Train rows: {train_mask.sum():,}  ({train_mask.mean()*100:.1f}%)')
print(f'Test rows:  {(~train_mask).sum():,}')

In [ ]:
df_full = add_historical_baseline(df_full, reference_df=df_full[train_mask])
print(f'hist_mean_occ nulls: {df_full["hist_mean_occ"].isnull().sum()}')

---
## Step 9 — Horizon expansion

Each (station, T) row becomes one row per horizon in HORIZONS.
Memory is kept under control by trimming to only the needed columns,
dowcasting float64 → float32, and processing one horizon at a time.

In [ ]:
print(f'Before expansion: {len(df_full):,} rows')
df_model = expand_horizons(df_full, horizons=HORIZONS)

del df_full
gc.collect()

print(f'After expansion:  {len(df_model):,} rows  ({len(HORIZONS)} horizons)')
print(f'Horizons present: {sorted(df_model["horizon_hours"].unique())}')

---
## Step 10 — Final checks and export

In [ ]:
from src.utils.config import FEATURES, TARGET

missing = [f for f in FEATURES if f not in df_model.columns]
if missing:
    print(f'WARNING — missing features: {missing}')
else:
    print('All expected features present.')

nulls = df_model[FEATURES + [TARGET]].isnull().sum()
nulls = nulls[nulls > 0]
if len(nulls):
    print(f'Nulls remaining:\n{nulls}')
else:
    print('No nulls in model features or target.')

In [ ]:
print(f'Final shape:  {df_model.shape}')
print(f'Stations:     {df_model["station_id"].nunique()}')
print(f'Date range:   {df_model["datetime_hour"].min()} → {df_model["datetime_hour"].max()}')
print()
print(df_model[FEATURES + [TARGET]].describe().round(3))

In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df_model.to_parquet(OUTPUT_PATH, index=False)

size_mb = OUTPUT_PATH.stat().st_size / 1e6
print(f'Saved: {OUTPUT_PATH}')
print(f'Size:  {size_mb:.1f} MB')